向量数据库的使用

---
- 向量库的数据增加
- 向量库的数据删除
- 向量库的相似性搜索
- 高级使用：MMR
- 高级使用：混合搜索

### 向量库的数据增加
---
统一使用国产嵌入模型

为了演示方便我们引入一个内存向量数据库，它将向量暂存在内存中，并使用字典以及numpy计算搜索的余弦相似度。

In [23]:
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv
import os

load_dotenv()

embeddings_model = OpenAIEmbeddings(
    model="Qwen/Qwen3-Embedding-8B",
    openai_api_key=os.getenv("api_key"),
    openai_api_base=os.getenv("base_url")
)

# 为了演示方便我们引入一个内存向量数据库，它将向量暂存在内存中，并使用字典以及numpy计算搜索的余弦相似度。
from langchain_core.vectorstores import InMemoryVectorStore
vector_store = InMemoryVectorStore(embedding=embeddings_model)

from langchain_core.documents import Document

document1 = Document(
    page_content = "今天在抖音学会了一个新菜:锅巴土豆泥!看起来简单，实际炸了厨房，连猫都嫌弃地走开了。",
    metadata = {"source": "社交媒体"}
)

document2 = Document(
    page_content = "小区遛狗大爷今日播报:广场舞大妈占领健身区，遛狗群众纷纷撤退。现场气氛诡异，BGM已循环播放《最炫民族风》两小时",
    metadata = {"source": "社区新闻"}
)

documents = [document1, document2]

vector_store.add_documents(documents=documents)

['b4f2d0b1-a8ee-4c10-91cf-cbd149fb50d7',
 '3ea93499-dc4e-45a8-a494-d62d11be1ab8']

你可以为添加的文档增加ID索引，便于后面管理


In [2]:
vector_store.add_documents(documents=documents, ids=["doc1", "doc2"])

['doc1', 'doc2']

### 向量库的删除
---

In [3]:
vector_store.delete(ids=["doc1"])

### 向量库的相似性搜索
---

In [4]:
query = "遛狗"
docs  = vector_store.similarity_search(query)
print(docs[0].page_content)

小区遛狗大爷今日播报:广场舞大妈占领健身区，遛狗群众纷纷撤退。现场气氛诡异，BGM已循环播放《最炫民族风》两小时


还可以使用"向量"查相似"向量"的方式来进行搜索

In [5]:
embedding_vector = embeddings_model.embed_query(query)
docs = vector_store.similarity_search_by_vector(embedding_vector)
print(docs[0].page_content)

小区遛狗大爷今日播报:广场舞大妈占领健身区，遛狗群众纷纷撤退。现场气氛诡异，BGM已循环播放《最炫民族风》两小时


注意：langchain只是在接口层面进行了封装，具体的搜索实现要依赖向量库本身的能力，比如Pinecone就可以进行元数据过滤，内存向量就不可以  

https://docs.pinecone.io/guides/get-started/quickstart   登录该网址->注册账号->申请API

In [ ]:
! pip uninstall pinecone-client pinecone -y
! pip cache purge
! pip install pinecone

Found existing installation: pinecone-client 6.0.0
Uninstalling pinecone-client-6.0.0:
  Successfully uninstalled pinecone-client-6.0.0
Found existing installation: pinecone 10.3.0
Uninstalling pinecone-10.3.0:
  Successfully uninstalled pinecone-10.3.0
Files removed: 4127 (11504.6 MB)
Directories removed: 6019
   ---------------------------------------- 0.0/3.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.0 MB ? eta -:--:--
   --- ------------------------------------ 0.3/3.0 MB ? eta -:--:--
   --- ------------------------------------ 0.3/3.0 MB ? eta -:--:--
   --- ------------------------------------ 0.3/3.0 MB ? eta -:--:

In [19]:
import getpass
from dotenv import load_dotenv
import os

load_dotenv()

from pinecone import Pinecone, ServerlessSpec

if not os.getenv("pinecone_api_key"):
    os.environ["pinecone_api_key"] = getpass.getpass("Enter your Pinecone API key: ")

pinecon_api_key = os.getenv("pinecone_api_key")

print(pinecon_api_key)

pc = Pinecone(api_key=pinecon_api_key)

pcsk_<REDACTED>


初始化

In [20]:
import time

index_name = "langchain-test-index"  # change if desired

existing_indexes = [index_info["name"] for index_info in pc.list_indexes()]

if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=4096,  # 注意维度要一致
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    while not pc.describe_index(index_name).status["ready"]:
        time.sleep(1)

index = pc.Index(index_name)

添加嵌入模型

In [21]:
from langchain_pinecone import PineconeVectorStore
# 因为国产嵌入模型的维度只有1024，所以无法使用
vector_store = PineconeVectorStore(index=index, embedding=OpenAIEmbeddings(model="text-embedding-3-large"))

ModuleNotFoundError: No module named 'langchain_pinecone'

添加内容

In [25]:
from uuid import uuid4
from langchain_core.documents import Document

document1 = Document(
    page_content = "今天早餐吃了老王家的生煎包，馅料实在得快从褶子里跳出来了!这才是真正的上海味道!",
    metadata = {"source": "tweet"}
)

document2 = Document(
    page_content = "明日天气预报:北京地区将出现大范围雾霾，建议市民戴好口罩，看不见脸的时候请不要慌张。",
    metadata = {"source": "news"}
)

document3 = Document(
    page_content = "终于搞定了AI聊天机器人!我问它'你是谁’，它回答'我是你爸爸’，看来还需要调教...",
    metadata = {"source": "tweet"}
)

document4 = Document(
    page_content = "震惊!本市一男子在便利店抢劫，只因店员说'扫码支付才有优惠’，现已被警方抓获。",
    metadata = {"source": "news"}
)

document5 = Document(
    page_content = "刚看完《流浪地球3》，特效简直炸裂!就是旁边大妈一直问'这是在哪拍的'有点影响观影体验",
    metadata = {"source": "tweet"}
)

document6 = Document(
    page_content = "新发布的小米14Ultra值不值得买?看完这篇测评你就知道为什么李老板笑得合不拢嘴了。",
    metadata = {"source": "website"}
)

document7 = Document(
    page_content = "2025年中超联赛十大最佳球员榜单新鲜出炉，第一名居然是他?!",
    metadata = {"source": "website"}
)

document8 = Document(
    page_content = "用LangChain开发的AI助手太神奇了!问它'人生的意义'，它给我推荐了一份外卖优惠券...",
    metadata = {"source": "tweet"}
)

document9 = Document(
    page_content = "A股今日暴跌，分析师称原因是'大家都在抢着卖'，投资者表示很有道理。",
    metadata = {"source": "news"}
)

document10 = Document(
    page_content = "感觉我马上要被删库跑路了，祝我好运/(ToT)/~~ ",
    metadata = {"source": "tweet"}
)

documents = [
    document1,
    document2,
    document3,
    document4,
    document5,
    document6,
    document7,
    document8,
    document9,
    document10,
]

uuids = [str(uuid4()) for _ in range(len(documents))] 

vector_store.add_documents(documents=documents, ids=uuids)

['b5dcb14e-c148-42fc-baf0-939c08057dc7',
 '13b99602-4914-48f6-bef7-7173960cbd10',
 'efe5c0d1-fb13-42db-bbe5-11226599c683',
 'be59971c-690c-40d4-8255-a3a03a7a766e',
 '663b9c1f-f320-4efb-b2a6-1bb7dff4a12f',
 'ff3f47a5-d7a7-45e6-bbd7-a704881a5777',
 'ee3556ab-469b-4e37-af00-34d9c4cd98b8',
 'ec1b1dc1-a723-4470-87c0-d0e2a0d436e8',
 '8e3b2626-9319-4732-94e4-da895bd3d3a9',
 '6b35996c-7645-4bd9-9611-8de50c846952']

删除最后一项

In [ ]:
vector_store.delete(ids=[uuids[-1]])

相似性搜索支持元数据过滤

In [ ]:
results = vector_store.similarity_search(
    "看电影",
    k=2,
    filter={"source", "tweet"},
)

for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

通过分数进行搜索

In [22]:
results = vector_store.similarity_search_with_score(
    "明天热么？", k=1, filter={"source": "news"}
)

for res, score in results:
    print(f"* [SIM={score:3f}] {res.page_content} [{res.metadata}]")

TypeError: 'dict' object is not callable

### MMR (最大边际相关性)
---
* 并非所有的向量库都支持

In [ ]:
vector_store.max_marginal_relevance_search(
    query="新手机",
    k=3,
    lambda_val=0.1,
    filter={"source": "website"},
)

[Document(id='61b60b43-7822-41a8-b1ee-45276f6aea87', metadata={'source': 'news'}, page_content="A股今日暴跌，分析师称原因是'大家都在抢着卖'，投资者表示很有道理。"),
 Document(id='9f3fd9b1-dc99-4d84-b613-4810fc55de1a', metadata={'source': 'tweet'}, page_content="刚看完《流浪地球3》，特效简直炸裂!就是旁边大妈一直问'这是在哪拍的'有点影响观影体验"),
 Document(id='6d3012bb-d53b-40b8-83af-5d37ade70297', metadata={'source': '社区新闻'}, page_content='小区遛狗大爷今日播报:广场舞大妈占领健身区，遛狗群众纷纷撤退。现场气氛诡异，BGM已循环播放《最炫民族风》两小时')]

In [40]:
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv
import os
from uuid import uuid4
from langchain_core.documents import Document

# 1. 加载环境变量
load_dotenv()

# 2. 初始化嵌入模型
embeddings_model = OpenAIEmbeddings(
    model="Qwen/Qwen3-Embedding-8B",
    openai_api_key=os.getenv("api_key"),
    openai_api_base=os.getenv("base_url")
)

# 3. 准备文档
document1 = Document(page_content="今天早餐吃了老王家的生煎包，馅料实在得快从褶子里跳出来了!这才是真正的上海味道!", metadata={"source": "tweet"})
document2 = Document(page_content="明日天气预报:北京地区将出现大范围雾霾，建议市民戴好口罩，看不见脸的时候请不要慌张。", metadata={"source": "news"})
document3 = Document(page_content="终于搞定了AI聊天机器人!我问它'你是谁’，它回答'我是你爸爸’，看来还需要调教...", metadata={"source": "tweet"})
document4 = Document(page_content="震惊!本市一男子在便利店抢劫，只因店员说'扫码支付才有优惠’，现已被警方抓获。", metadata={"source": "news"})
document5 = Document(page_content="刚看完《流浪地球3》，特效简直炸裂!就是旁边大妈一直问'这是在哪拍的'有点影响观影体验", metadata={"source": "tweet"})
document6 = Document(page_content="新发布的小米14Ultra值不值得买?看完这篇测评你就知道为什么李老板笑得合不拢嘴了。", metadata={"source": "website"})
document7 = Document(page_content="2025年中超联赛十大最佳球员榜单新鲜出炉，第一名居然是他?!", metadata={"source": "website"})
document8 = Document(page_content="用LangChain开发的AI助手太神奇了!问它'人生的意义'，它给我推荐了一份外卖优惠券...", metadata={"source": "tweet"})
document9 = Document(page_content="A股今日暴跌，分析师称原因是'大家都在抢着卖'，投资者表示很有道理。", metadata={"source": "news"})
document10 = Document(page_content="感觉我马上要被删库跑路了，祝我好运/(ToT)/~~ ", metadata={"source": "tweet"})

documents = [document1, document2, document3, document4, document5, document6, document7, document8, document9, document10]
uuids = [str(uuid4()) for _ in range(len(documents))]

# 4. 使用 Chroma 向量存储（支持过滤）
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=documents,  # ✅ 修正变量名
    embedding=embeddings_model,  # ✅ 修正变量名
    ids=uuids
)

# 5. 执行带过滤的 MMR 搜索
results = vector_store.max_marginal_relevance_search(
    query="新手机",
    k=3,
    lambda_mult=0.1,
    filter={"source": "website"}  # ✅ Chroma 支持过滤
)

# 6. 打印结果
print(f"找到 {len(results)} 个相关文档：\n")
for i, doc in enumerate(results, 1):
    print(f"{i}. [{doc.metadata['source']}] {doc.page_content}")

找到 3 个相关文档：

1. [website] 新发布的小米14Ultra值不值得买?看完这篇测评你就知道为什么李老板笑得合不拢嘴了。
2. [website] 新发布的小米14Ultra值不值得买?看完这篇测评你就知道为什么李老板笑得合不拢嘴了。
3. [website] 2025年中超联赛十大最佳球员榜单新鲜出炉，第一名居然是他?!


In [41]:
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv
import os
from uuid import uuid4
from langchain_core.documents import Document
from langchain_chroma import Chroma  # 确保已安装: pip install langchain-chroma chromadb

load_dotenv()

# 1. 初始化嵌入模型
embeddings_model = OpenAIEmbeddings(
    model="Qwen/Qwen3-Embedding-8B",
    openai_api_key=os.getenv("api_key"),
    openai_api_base=os.getenv("base_url")
)

# 2. 准备文档
document1 = Document(page_content="今天早餐吃了老王家的生煎包，馅料实在得快从褶子里跳出来了!这才是真正的上海味道!", metadata={"source": "tweet"})
document2 = Document(page_content="明日天气预报:北京地区将出现大范围雾霾，建议市民戴好口罩，看不见脸的时候请不要慌张。", metadata={"source": "news"})
document3 = Document(page_content="终于搞定了AI聊天机器人!我问它'你是谁’，它回答'我是你爸爸’，看来还需要调教...", metadata={"source": "tweet"})
document4 = Document(page_content="震惊!本市一男子在便利店抢劫，只因店员说'扫码支付才有优惠’，现已被警方抓获。", metadata={"source": "news"})
document5 = Document(page_content="刚看完《流浪地球3》，特效简直炸裂!就是旁边大妈一直问'这是在哪拍的'有点影响观影体验", metadata={"source": "tweet"})
document6 = Document(page_content="新发布的小米14Ultra值不值得买?看完这篇测评你就知道为什么李老板笑得合不拢嘴了。", metadata={"source": "website"})
document7 = Document(page_content="2025年中超联赛十大最佳球员榜单新鲜出炉，第一名居然是他?!", metadata={"source": "website"})
document8 = Document(page_content="用LangChain开发的AI助手太神奇了!问它'人生的意义'，它给我推荐了一份外卖优惠券...", metadata={"source": "tweet"})
document9 = Document(page_content="A股今日暴跌，分析师称原因是'大家都在抢着卖'，投资者表示很有道理。", metadata={"source": "news"})
document10 = Document(page_content="感觉我马上要被删库跑路了，祝我好运/(ToT)/~~ ", metadata={"source": "tweet"})

documents = [document1, document2, document3, document4, document5, document6, document7, document8, document9, document10]
uuids = [str(uuid4()) for _ in range(len(documents))] 

# 3. 创建 Chroma 向量存储 (直接从文档初始化)
# 注意：这里不需要先创建 InMemoryVectorStore
vector_store = Chroma.from_documents(
    documents=documents,       # ✅ 使用正确的变量名 documents
    embedding=embeddings_model, # ✅ 使用正确的变量名 embeddings_model
    ids=uuids                  # 可选：传入自定义 ID
)

# 4. 执行带过滤的 MMR 搜索
print("正在搜索 source='website' 且与 '新手机' 相关的文档...")
results = vector_store.max_marginal_relevance_search(
    query="新手机",
    k=3,
    lambda_mult=0.1,
    filter={"source": "website"}  # ✅ Chroma 支持过滤
)

# 5. 打印结果
for i, doc in enumerate(results):
    print(f"\n--- 结果 {i+1} ---")
    print(f"内容: {doc.page_content}")
    print(f"来源: {doc.metadata['source']}")

正在搜索 source='website' 且与 '新手机' 相关的文档...

--- 结果 1 ---
内容: 新发布的小米14Ultra值不值得买?看完这篇测评你就知道为什么李老板笑得合不拢嘴了。
来源: website

--- 结果 2 ---
内容: 新发布的小米14Ultra值不值得买?看完这篇测评你就知道为什么李老板笑得合不拢嘴了。
来源: website

--- 结果 3 ---
内容: 2025年中超联赛十大最佳球员榜单新鲜出炉，第一名居然是他?!
来源: website


# 混合搜索
---

In [52]:
! pip install rank_bm25

In [48]:
import os

api_key = os.getenv("pinecone_api_key")

In [49]:
from langchain_community.retrievers import (
    PineconeHybridSearchRetriever,
)

初始化

In [ ]:
import os

from pinecone import Pinecone, ServerlessSpec

from langchain_openai import OpenAIEmbeddings

# 1. 初始化嵌入模型
embeddings_model = OpenAIEmbeddings(
    model="Qwen/Qwen3-Embedding-8B",
    openai_api_key=os.getenv("api_key"),
    openai_api_base=os.getenv("base_url")
)

index_name = "langchain-pinecone-hybrid-search"

# initialize Pinecone client
pc = Pinecone(api_key=api_key)

# create the index
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=1536, # dimensionality of dense model
        metric="dotproduct", # sparse values supported only for dotproduct
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

from pinecone_text.sparse import BM25Encoder

bm25_encoder = BM25Encoder().default()

corpus = ["foo", "bar", "world", "hello"]

bm25_encoder.fit(corpus)

bm25_encoder.dump("bm25_values.json")

bm25_encoder = BM25Encoder().load("bm25_values.json")

retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings_model, sparse_encoder=bm25_encoder, index=index
)


添加文本

In [ ]:
retriever.add_texts(["foo", "bar", "world", "hello"])

检索

In [ ]:
result = retriever.invoke("foo")
result[0]

In [56]:
# 兼容 python 3.14 的版本
import os
import json
import numpy as np
from collections import Counter
from typing import List
from dataclasses import dataclass

from pinecone import Pinecone, ServerlessSpec
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from dotenv import load_dotenv

load_dotenv()

# ============================================
# 1. 初始化嵌入模型和 Pinecone
# ============================================
embeddings_model = OpenAIEmbeddings(
    model="Qwen/Qwen3-Embedding-8B",
    openai_api_key=os.getenv("api_key"),
    openai_api_base=os.getenv("base_url")
)

index_name = "langchain-pinecone-hybrid-search"

# 删除旧索引
if index_name in pc.list_indexes().names():
    pc.delete_index(index_name)
    print(f"✅ 已删除旧索引: {index_name}")
    
api_key = os.getenv("pinecone_api_key")
pc = Pinecone(api_key=api_key)

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=4096,
        metric="dotproduct",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

# ============================================
# 2. 自定义 BM25 编码器（替代 pinecone-text）
# ============================================
class CustomBM25Encoder:
    """兼容 Python 3.14 的 BM25 编码器"""
    
    def __init__(self):
        self.idf = {}
        self.avg_doc_len = 0
        self.vocab = {}
        
    def _tokenize(self, text):
        """简单分词（生产环境建议用 jieba）"""
        return list(text.replace(" ", ""))
    
    def fit(self, corpus):
        """训练 BM25 模型"""
        doc_count = len(corpus)
        doc_freq = Counter()
        total_len = 0
        
        for doc in corpus:
            tokens = self._tokenize(doc)
            total_len += len(tokens)
            for token in set(tokens):
                doc_freq[token] += 1
        
        for token, freq in doc_freq.items():
            self.idf[token] = np.log((doc_count - freq + 0.5) / (freq + 0.5) + 1)
        
        self.avg_doc_len = total_len / doc_count
        self.vocab = {token: idx for idx, token in enumerate(self.idf.keys())}
        
    def encode_documents(self, text):
        """编码文档为稀疏向量"""
        tokens = self._tokenize(text)
        token_freq = Counter(tokens)
        
        indices, values = [], []
        for token, freq in token_freq.items():
            if token in self.idf:
                idx = self.vocab[token]
                idf = self.idf[token]
                tf = (freq * 1.5) / (freq + 1.5 * (1 - 0.75 + 0.75 * len(tokens) / self.avg_doc_len))
                indices.append(idx)
                values.append(float(idf * tf))
        
        return {"indices": indices, "values": values}
    
    def encode_queries(self, text):
        """编码查询为稀疏向量"""
        return self.encode_documents(text)
    
    def dump(self, filepath):
        """保存模型"""
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump({"idf": self.idf, "avg_doc_len": self.avg_doc_len, "vocab": self.vocab}, f, ensure_ascii=False)
    
    def load(self, filepath):
        """加载模型"""
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)
        self.idf, self.avg_doc_len, self.vocab = data["idf"], data["avg_doc_len"], data["vocab"]
        return self

# 初始化 BM25
bm25_encoder = CustomBM25Encoder()
corpus = ["foo", "bar", "world", "hello"]
bm25_encoder.fit(corpus)
bm25_encoder.dump("bm25_values.json")
bm25_encoder = CustomBM25Encoder().load("bm25_values.json")

# ============================================
# 3. 自定义 Hybrid Search Retriever（替代 PineconeHybridSearchRetriever）
# ============================================
class PineconeHybridSearchRetriever:
    """手动实现的混合搜索检索器"""
    
    def __init__(self, embeddings, sparse_encoder, index, alpha=0.5):
        self.embeddings = embeddings
        self.sparse_encoder = sparse_encoder
        self.index = index
        self.alpha = alpha  # 混合权重：alpha * dense + (1-alpha) * sparse
        
    def add_texts(self, texts: List[str]):
        """添加文本到向量库"""
        # 生成稠密向量
        dense_vecs = self.embeddings.embed_documents(texts)
        
        # 生成稀疏向量
        sparse_vecs = [self.sparse_encoder.encode_documents(text) for text in texts]
        
        # 组装并写入
        records = []
        for i, text in enumerate(texts):
            records.append({
                "id": f"doc_{i}",
                "values": dense_vecs[i],
                "sparse_values": sparse_vecs[i],
                "metadata": {"text": text}
            })
        
        self.index.upsert(vectors=records)
        print(f"✅ 已添加 {len(texts)} 条文档")
        
    def invoke(self, query: str, top_k: int = 4) -> List[Document]:
        """执行混合搜索"""
        # 生成查询向量
        dense_query = self.embeddings.embed_query(query)
        sparse_query = self.sparse_encoder.encode_queries(query)
        
        # 执行混合查询
        results = self.index.query(
            vector=dense_query,
            sparse_vector=sparse_query,
            top_k=top_k,
            include_metadata=True
        )
        
        # 转换为 LangChain Document 格式
        docs = [
            Document(
                page_content=match['metadata']['text'],
                metadata={"score": match['score'], "id": match['id']}
            )
            for match in results['matches']
        ]
        
        return docs

# ============================================
# 4. 使用示例
# ============================================
retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings_model,
    sparse_encoder=bm25_encoder,
    index=index,
    alpha=0.5  # 稠密和稀疏的混合权重
)

# 添加文档
retriever.add_texts(["foo", "bar", "world", "hello"])

# 检索
result = retriever.invoke("foo")
print(f"\n🔍 查询 'foo' 的结果:")
print(f"最匹配的文档: {result[0].page_content}")
print(f"得分: {result[0].metadata['score']:.4f}")

# 显示所有结果
print(f"\n📋 所有结果:")
for i, doc in enumerate(result):
    print(f"{i+1}. {doc.page_content} (得分: {doc.metadata['score']:.4f})")

✅ 已删除旧索引: langchain-pinecone-hybrid-search
✅ 已添加 4 条文档

🔍 查询 'foo' 的结果:
最匹配的文档: foo
得分: 1.7758

📋 所有结果:
1. foo (得分: 1.7758)
2. world (得分: 0.9842)
3. hello (得分: 0.9745)
4. bar (得分: 0.8820)
